# 04 — Gold: ML feature store

Fits and persists the feature pipeline (StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler),
writes the Gold feature-vector table, and stores the pipeline model in the artifacts volume
so scoring reuses the exact same encoding.

Target: `label = 1` if `arrival_delay >= 15` else `0`. Cancelled / diverted (null delay) is filtered here.

In [0]:
import sys
sys.path.append("..")

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler,
)
from pyspark.sql.functions import col, when

from src import config

silver = spark.table(config.SILVER).filter(col("arrival_delay").isNotNull())
labeled = silver.withColumn(
    "label",
    when(col("arrival_delay") >= config.DELAY_THRESHOLD_MINUTES, 1.0).otherwise(0.0),
)

print(f"Silver rows (non-cancelled): {labeled.count():,}")
positive_rate = labeled.filter(col("label") == 1.0).count() / labeled.count()
print(f"Positive class rate: {positive_rate:.3%}")

Silver rows (non-cancelled): 2,463,979
Positive class rate: 19.878%


## Feature groups

In [0]:
categorical_cols = [
    "airline_name", "airline_code", "origin_airport_code",
    "destination_airport_code", "season",
]
boolean_cols = ["is_weekend", "is_holiday", "is_near_holiday", "is_holiday_period"]
numerical_cols = [
    "flight_month", "flight_year", "day_of_week", "week_of_year", "day_of_month",
    "quarter", "fl_number", "crs_elapsed_time", "distance", "dep_hour", "arr_hour",
    "dep_delay",
]

## Build + fit the pipeline

In [0]:
# Check if pipeline was already fitted in this session (to avoid ML cache overflow)
try:
    # If pipeline_model exists, reuse it instead of fitting again
    _ = pipeline_model
    print("Reusing existing pipeline_model from previous run (avoids ML cache overflow)")
    # Still need to transform the data
    prepared = labeled.na.fill(0, subset=numerical_cols + boolean_cols)
    for c in categorical_cols:
        prepared = prepared.na.fill("UNKNOWN", subset=[c])
    # Avoid duplicate columns: label is already in labeled.columns
    gold = pipeline_model.transform(prepared).select(
        "label", "features", *[c for c in labeled.columns if c != "label"],
    )
except NameError:
    # First run: fit a new pipeline
    print("Fitting new pipeline (first run in this session)")
    
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in categorical_cols
    ]
    encoders = [
        OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
        for c in categorical_cols
    ]

    assembled_cols = (
        numerical_cols
        + boolean_cols
        + [f"{c}_ohe" for c in categorical_cols]
    )

    assembler = VectorAssembler(
        inputCols=assembled_cols,
        outputCol="features_raw",
        handleInvalid="keep",
    )
    scaler = StandardScaler(
        inputCol="features_raw", outputCol="features", withMean=False, withStd=True,
    )

    pipeline = Pipeline(stages=[*indexers, *encoders, assembler, scaler])

    # Cheap fill for numerics before fitting.
    prepared = labeled.na.fill(0, subset=numerical_cols + boolean_cols)
    for c in categorical_cols:
        prepared = prepared.na.fill("UNKNOWN", subset=[c])

    pipeline_model = pipeline.fit(prepared)
    # Avoid duplicate columns: label is already in labeled.columns
    gold = pipeline_model.transform(prepared).select(
        "label", "features", *[c for c in labeled.columns if c != "label"],
    )

    # Release intermediate objects to help with ML cache management
    import gc
    del indexers, encoders, assembler, scaler, pipeline, prepared
    gc.collect()

print(f"Gold DataFrame created with {len(gold.columns)} columns")

Reusing existing pipeline_model from previous run (avoids ML cache overflow)
Gold DataFrame created with 27 columns


## Persist Gold + pipeline model

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.GOLD)
)
gold_count = spark.table(config.GOLD).count()
print(f"Gold rows: {gold_count:,}")

pipeline_path = f"{config.ARTIFACT_VOLUME}/feature_pipeline"
pipeline_model.write().overwrite().save(pipeline_path)
print(f"Feature pipeline saved to {pipeline_path}")

Gold rows: 2,463,979
Feature pipeline saved to /Volumes/workspace/flights/artifacts/feature_pipeline


## Feature manifest
The list of assembled input columns is written to a small Delta table so scoring
reconstructs the exact same feature space without hardcoding a 40-integer array.

In [0]:
from pyspark.sql import Row

def _group_for(name: str) -> str:
    if name in numerical_cols:
        return "numeric"
    if name in boolean_cols:
        return "boolean"
    return "one_hot"

manifest_rows = [
    Row(position=i, feature_group=_group_for(c), source_column=c)
    for i, c in enumerate(assembled_cols)
]
manifest_df = spark.createDataFrame(manifest_rows)
(
    manifest_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(config.FEATURE_MANIFEST)
)
print(f"Wrote feature manifest ({len(manifest_rows)} rows) → {config.FEATURE_MANIFEST}")

Wrote feature manifest (21 rows) → workspace.flights.feature_manifest


In [0]:
%sql
DESCRIBE HISTORY workspace.flights.gold_ml_features;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-30T08:28:38.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1485683378104422),83de254c-59b5-4b87-88be-4dd7fb24588c,0830-070804-aa8kww6x-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 113937062, numDeletionVectorsRemoved -> 0, numOutputRows -> 2463979, numOutputBytes -> 113937062)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-30T08:27:05.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1485683378104422),1ae6dd66-81f4-4f0d-bc2e-4eb195fc649d,0830-070804-aa8kww6x-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 113937062, numDeletionVectorsRemoved -> 0, numOutputRows -> 2463979, numOutputBytes -> 113937062)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
1,2026-08-30T08:03:24.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1485683378104422),d0aa703b-9e28-4ba3-ac77-f9dc81dd6f88,0830-070804-aa8kww6x-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 113937062, numDeletionVectorsRemoved -> 0, numOutputRows -> 2463979, numOutputBytes -> 113937062)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-30T07:45:05.000Z,71506073265303,omitelkady1@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1485683378104422),703dffe0-2e5e-4704-9d12-476ef0421065,0830-070804-aa8kww6x-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 2463979, numOutputBytes -> 113937062)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
